#Encoding Categorical Features
>Categorical encoding is the process of converting categorical data (text or labels) into numerical representations that machine learning algorithms can use.


##Setup

In [86]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

##Load Dataset

In [87]:
import kagglehub
import shutil

# Download latest version
path = kagglehub.dataset_download("tomalislam/titanic-dataset-fot-practice")
print("Path to dataset files:", path)

# Copy ignoring hidden NFS lock files to prevent Permission Denied errors
shutil.copytree(path, '/content/dataset', dirs_exist_ok=True, ignore=shutil.ignore_patterns('.nfs*'))

Using Colab cache for faster access to the 'titanic-dataset-fot-practice' dataset.
Path to dataset files: /kaggle/input/titanic-dataset-fot-practice


'/content/dataset'

In [88]:
dfTest = pd.read_csv('/content/dataset/imputedTest.csv')
dfTrain = pd.read_csv('/content/dataset/imputedTrain.csv')

##Split data

In [89]:
target = 'survived'
xTrain = dfTrain.drop(target,axis=1)
xTest = dfTest.drop(target,axis=1)
yTrain = dfTrain[target]
yTest = dfTest[target]

In [90]:
print(f'Test Shape :{dfTest.shape}')
print(f'Train Shape :{dfTrain.shape}')
print(f'feature train shape : {xTrain.shape}')
print(f'output train shape :  {yTrain.shape}')
print(f'feature test shape : {xTest.shape}')
print(f'output test shape :  {yTest.shape}')

Test Shape :(179, 14)
Train Shape :(712, 14)
feature train shape : (712, 13)
output train shape :  (712,)
feature test shape : (179, 13)
output test shape :  (179,)


##Label Encoding

In [91]:
xTrain.drop(['embark_town','class','alive'],inplace=True,axis=1)
xTest.drop(['embark_town','class','alive'],inplace=True,axis=1)

In [92]:
print(xTrain.columns.to_list())
print(xTest.columns.to_list())

['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'who', 'adult_male', 'alone']
['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'who', 'adult_male', 'alone']


##Filter Numerical And Categorical Columns

In [93]:
numCol = ["age", "sibsp", "parch", "fare"]
catCol = ['pclass', 'sex', 'embarked', 'who', 'adult_male', 'alone']

print("Numerical columns:", numCol)
print("Categorical columns:", catCol)

Numerical columns: ['age', 'sibsp', 'parch', 'fare']
Categorical columns: ['pclass', 'sex', 'embarked', 'who', 'adult_male', 'alone']


In [94]:
xTrain.head()

,pclass,sex,age,sibsp,parch,fare,embarked,who,adult_male,alone
0,3,male,28.5,0,0,56.4958,S,man,True,True
1,2,male,28.5,0,0,0.0000,S,man,True,True
2,1,male,28.5,0,0,221.7792,S,man,True,True
3,3,female,18.0,0,1,9.3500,S,woman,False,False
4,2,female,31.0,1,1,26.2500,S,woman,False,False


In [95]:
print('Train')
for col in catCol:
  print(f'{col} -> {xTrain[col].unique()}')
  print('-'*50)
print('Test')
for col in catCol:
  print(f'{col} -> {xTest[col].unique()}')
  print('-'*50)


Train
pclass -> [3 2 1]
--------------------------------------------------
sex -> ['male' 'female']
--------------------------------------------------
embarked -> ['S' 'C' 'Q']
--------------------------------------------------
who -> ['man' 'woman' 'child']
--------------------------------------------------
adult_male -> [ True False]
--------------------------------------------------
alone -> [ True False]
--------------------------------------------------
Test
pclass -> [3 2 1]
--------------------------------------------------
sex -> ['male' 'female']
--------------------------------------------------
embarked -> ['S' 'C' 'Q']
--------------------------------------------------
who -> ['man' 'woman' 'child']
--------------------------------------------------
adult_male -> [ True False]
--------------------------------------------------
alone -> [False  True]
--------------------------------------------------


| Column       | Type     | Explanation |
|--------------|----------|-------------|
| **pclass**       | Ordinal | Travel classes have a natural order (1 > 2 > 3) |
| **sex**          | Nominal (binary) | No ranking between male and female |
| **embarked**     | Nominal | Ports do not have any order |
| **who**          | Nominal *(optional ordinal)* | Usually treated as nominal unless modeling age groups |
| **adult_male**   | Nominal (binary) | Binary indicator, no inherent order |
| **deck**         | Nominal | Cabin letters do not represent a meaningful order |
| **alone**        | Nominal (binary) | Binary indicator, no hierarchy |

In [96]:
labelEncodingCol = ['sex','adult_male','alone']
oneHotCol = ['embarked','who']

In [97]:
xTrainEn = xTrain.copy(deep=True)
xTestEn = xTest.copy(deep=True)

##Label Encoding
**IMPORTANT: The encoders (labelEncoderStore) will be saved for later prediction use on new data**

In [98]:
from sklearn.preprocessing import LabelEncoder
labelEncoderStore = {}
for col in labelEncodingCol:
  le = LabelEncoder()
  xTrainEn[col] = le.fit_transform(xTrainEn[col])
  labelEncoderStore[col] = le

  xTestEn[col] = le.transform(xTestEn[col])

In [99]:
print(xTrainEn.head())
print('-'*50)
print(xTestEn.head())

   pclass  sex   age  sibsp  parch      fare embarked    who  adult_male  \
0       3    1  28.5      0      0   56.4958        S    man           1   
1       2    1  28.5      0      0    0.0000        S    man           1   
2       1    1  28.5      0      0  221.7792        S    man           1   
3       3    0  18.0      0      1    9.3500        S  woman           0   
4       2    0  31.0      1      1   26.2500        S  woman           0   

   alone  
0      1  
1      1  
2      1  
3      0  
4      0  
--------------------------------------------------
   pclass  sex   age  sibsp  parch     fare embarked    who  adult_male  alone
0       3    1  24.0      2      0  24.1500        S    man           1      0
1       3    1  44.0      0      1  16.1000        S    man           1      0
2       3    1  22.0      0      0   7.2250        C    man           1      1
3       3    1  41.0      2      0  14.1083        S    man           1      0
4       3    0  28.5      1    

##One Hot Encoding

In [100]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(drop='first',handle_unknown='ignore',sparse_output=False)
oneHotTraining = ohe.fit_transform(xTrainEn[oneHotCol])
oneHotTesting = ohe.transform(xTestEn[oneHotCol])
cols = ohe.get_feature_names_out(oneHotCol)
oneHotTrainDf = pd.DataFrame(oneHotTraining,columns=cols)
oneHotTestDf = pd.DataFrame(oneHotTesting,columns=cols)

In [101]:
oneHotTestDf.head()


,embarked_Q,embarked_S,who_man,who_woman
0,0.0,1.0,1.0,0.0
1,0.0,1.0,1.0,0.0
2,0.0,0.0,1.0,0.0
3,0.0,1.0,1.0,0.0
4,1.0,0.0,0.0,1.0


In [102]:
xTestEn.drop(oneHotCol,axis=1,inplace=True)
xTrainEn.drop(oneHotCol,axis=1,inplace=True)

In [103]:
xTestEn = pd.concat([xTestEn,oneHotTestDf],axis=1)
xTrainEn = pd.concat([xTrainEn,oneHotTrainDf],axis=1)

In [104]:
xTestEn.head()

,pclass,sex,age,sibsp,parch,fare,adult_male,alone,embarked_Q,embarked_S,who_man,who_woman
0,3,1,24.0,2,0,24.1500,1,0,0.0,1.0,1.0,0.0
1,3,1,44.0,0,1,16.1000,1,0,0.0,1.0,1.0,0.0
2,3,1,22.0,0,0,7.2250,1,1,0.0,0.0,1.0,0.0
3,3,1,41.0,2,0,14.1083,1,0,0.0,1.0,1.0,0.0
4,3,0,28.5,1,0,15.5000,0,0,1.0,0.0,0.0,1.0


In [105]:
xTestEn.dtypes

,0
pclass,int64
sex,int64
age,float64
sibsp,int64
parch,int64
fare,float64
adult_male,int64
alone,int64
embarked_Q,float64
embarked_S,float64


In [106]:
print(xTestEn.shape)
print(xTrainEn.shape)

(179, 12)
(712, 12)


In [107]:
testEncoded = xTestEn.copy(deep=True)
testEncoded[target] = yTest
trainEncoded = xTrainEn.copy(deep=True)
trainEncoded[target] = yTrain


In [108]:
testEncoded.head()

,pclass,sex,age,sibsp,parch,fare,adult_male,alone,embarked_Q,embarked_S,who_man,who_woman,survived
0,3,1,24.0,2,0,24.1500,1,0,0.0,1.0,1.0,0.0,0
1,3,1,44.0,0,1,16.1000,1,0,0.0,1.0,1.0,0.0,0
2,3,1,22.0,0,0,7.2250,1,1,0.0,0.0,1.0,0.0,1
3,3,1,41.0,2,0,14.1083,1,0,0.0,1.0,1.0,0.0,0
4,3,0,28.5,1,0,15.5000,0,0,1.0,0.0,0.0,1.0,1


In [111]:
print(trainEncoded.shape)
print(testEncoded.shape)

(712, 13)
(179, 13)


In [109]:
testEncoded.to_csv('/content/dataset/testEncoded.csv',index=False)
trainEncoded.to_csv('/content/dataset/trainEncoded.csv',index=False)